# Test lc_multi_outputs Extension

This notebook tests that the lc_multi_outputs JupyterLab extension is properly installed and enabled.

## Parameters

In [ ]:
# Default parameters (will be overridden by Papermill)
jupyter_url = "http://localhost:8888/lab"
jupyter_token = "5099a139f1164f6394eb77421041a30a"
default_result_path = None
close_on_fail = False
transition_timeout = 30000

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
print(f"Created work directory: {work_dir}")

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## Open JupyterLab and wait for it to load

In [ ]:
async def _step_wait_for_loading(page):
    await page.goto(f"{jupyter_url}?token={jupyter_token}")

    # Wait for JupyterLab to load
    await expect(page.locator('#jp-main-dock-panel')).to_be_visible(timeout=transition_timeout)

    # Wait for the browser to be visible
    await expect(page.locator('.jp-DirListing')).to_be_visible(timeout=transition_timeout)

await run_pw(_step_wait_for_loading)

## Create a new notebook for testing multi_outputs

In [ ]:
async def _step_create_notebook(page):
    # Use the Launcher tab to create a new notebook
    # This avoids the kernel selection dialog that appears with File > New > Notebook
    
    # Wait for Launcher tab to be visible
    launcher_tabs = page.locator('.lm-TabBar-tab').filter(has_text='Launcher')
    launcher_tabs_count = await launcher_tabs.count()

    if launcher_tabs_count > 0:
        launcher_tab = launcher_tabs.first
    else:
        # Execute File > New Launcher using Shift+Enter
        await page.keyboard.press('Shift+ControlOrMeta+L')
        launcher_tab = launcher_tabs.first
    
    await expect(launcher_tab).to_be_visible(timeout=transition_timeout)
    
    # Click on Launcher tab to ensure it's active
    await launcher_tab.click()
    
    # Wait for launcher content to be visible
    await expect(page.locator('.jp-Launcher')).to_be_visible(timeout=transition_timeout)
    
    # Click on Python notebook launcher card
    # Using data-category attribute for stability
    launcher_card = page.locator('.jp-LauncherCard[data-category="Notebook"]').first
    await expect(launcher_card).to_be_visible(timeout=transition_timeout)
    await launcher_card.click()
    
    # Wait for the notebook panel to be created and loaded
    await expect(page.locator('.jp-NotebookPanel')).to_be_visible(timeout=transition_timeout)
    
    # Wait for the first cell to be ready
    await expect(page.locator('.jp-Cell.jp-CodeCell')).to_be_visible(timeout=transition_timeout)
    
    print("✓ New notebook created via Launcher")

await run_pw(_step_create_notebook)

## Execute a cell with output to trigger multi_outputs

In [ ]:
async def _step_execute_cell(page):
    # Get the first code cell
    code_cell = page.locator('.jp-Cell.jp-CodeCell').first
    await expect(code_cell).to_be_visible(timeout=transition_timeout)
    
    # Click on the cell editor to focus it
    editor = code_cell.locator('.cm-content')
    await editor.click()
    
    # Type some code that produces output
    # Use a simple print statement that will show in the output area
    await page.keyboard.type('print("Test output1 for multi_outputs extension")')
    
    # Execute the cell using Shift+Enter
    await page.keyboard.press('Shift+Enter')
    
    # Wait for the output to appear in the output area
    output = code_cell.locator('.jp-OutputArea-output')
    await expect(output).to_be_visible(timeout=transition_timeout)
    
    # Wait a bit for the multi_outputs extension to add its UI elements
    await page.wait_for_timeout(1000)
    
    print("✓ Cell executed with output")

await run_pw(_step_execute_cell)

## Test multi_outputs basic functionality (pin button and output saving)

In [ ]:
async def _step_verify_multi_outputs(page):
    # Click the first cell
    code_cell = page.locator('.jp-Cell.jp-CodeCell').first
    await code_cell.click()

    # Check that the output area exists and is visible
    output_area = code_cell.locator('.jp-OutputArea')
    await expect(output_area).to_be_visible(timeout=transition_timeout)
    
    # Check that the output is rendered properly
    output_text = await output_area.locator('.jp-OutputArea-output').text_content()
    assert "Test output1" in output_text, f"Expected output not found. Got: {output_text}"

    # click pin button on cell
    output_ui = output_area.locator('.multi-outputs-ui')
    pin_button = output_ui.locator('button')
    await expect(pin_button).to_be_visible(timeout=transition_timeout)
    await pin_button.click()

    # Check that there are tabs
    multi_outputs_tabs_container = code_cell.locator('.multi-output-container')
    await expect(multi_outputs_tabs_container).to_be_visible(timeout=transition_timeout)

    # Click on the cell editor to focus it
    editor = code_cell.locator('.cm-content')
    await editor.click()

    # Click unfreeze button if it exists (LC_run_through extension)
    # This ensures the first cell is not frozen before we try to edit it
    unfreeze_button = page.locator('.run-through-toolbar-button__unfreeze')

    # Check if the button exists
    button_count = await unfreeze_button.count()

    if button_count > 0:
        await unfreeze_button.click()
        print("✓ Clicked unfreeze button")
    else:
        print("✓ No unfreeze button found (cells not frozen)")

    # Clear all text in the cell before typing new content
    # Use ControlOrMeta+A (works on all platforms: Ctrl+A on Linux/Windows, Command+A on Mac)
    await editor.click()
    await page.keyboard.press('ControlOrMeta+A')
    await page.keyboard.press('Delete')
    print("✓ Cleared cell content")
    
    # Type some code that produces output
    # Use a simple print statement that will show in the output area
    await page.keyboard.type('print("Test output2 for multi_outputs extension")')

    # Execute the cell using Shift+Enter
    await page.keyboard.press('Shift+Enter')

    # Wait a bit for the multi_outputs extension to add its UI elements
    await page.wait_for_timeout(1000)

    # Click the first cell
    await code_cell.click()

    # click pin button on toolbar
    toolbar_pin_button = page.locator('jp-button[title="Pin Outputs"]')
    await expect(toolbar_pin_button).to_be_visible(timeout=transition_timeout)
    await toolbar_pin_button.click()

    # has tabs
    tab_output_1 = multi_outputs_tabs_container.locator('li[id="tab-output-1"]')
    await expect(tab_output_1).to_be_visible(timeout=transition_timeout)
    tab_output_2 = multi_outputs_tabs_container.locator('li[id="tab-output-2"]')
    await expect(tab_output_2).to_be_visible(timeout=transition_timeout)

    # Check that the output (output2) is rendered properly
    output_area_2 = multi_outputs_tabs_container.locator('div[id="output-2"]')
    output_text = await output_area_2.locator('.jp-OutputArea-output').text_content()
    assert "Test output2" in output_text, f"Expected output not found. Got: {output_text}"

    # Check that the output (output1) is rendered properly
    await tab_output_1.click()
    output_area_1 = multi_outputs_tabs_container.locator('div[id="output-1"]')
    output_text = await output_area_1.locator('.jp-OutputArea-output').text_content()
    assert "Test output1" in output_text, f"Expected output not found. Got: {output_text}"
    
    print("✓ Multi_outputs pin button and output saving functionality verified")

await run_pw(_step_verify_multi_outputs)

## Delete the notebook for testing

In [ ]:
async def _step_delete_notebook(page):
    # Get selected tab
    dockpanel_tabbar = page.locator('.lm-DockPanel-tabBar')
    selected_tab = dockpanel_tabbar.locator('li[aria-selected="true"]')

    # Get file name
    tab_label = selected_tab.locator('.lm-TabBar-tabLabel')
    file_name = await tab_label.text_content()
    print(f"File name of the notebook for testing is '{file_name}'")

    # Click close icon
    close_icon = selected_tab.locator('.lm-TabBar-tabCloseIcon')
    await close_icon.click()

    # Click discard button on dialog
    dialog = page.locator('.lm-Widget.lm-Panel.jp-Dialog-content')
    discard_button = dialog.locator('.jp-Dialog-button.jp-mod-accept.jp-mod-warn')
    await discard_button.click()

    # Right click on file name
    file_browser_panel = page.locator('.jp-FileBrowser-Panel')
    file_item = file_browser_panel.locator(f'.jp-DirListing-itemText:has(:text("{file_name}"))')
    li_elem = file_item.locator('xpath=ancestor::li[1]')
    await li_elem.click(button="right")

    # Click Delete menu
    context_menu = page.locator('.lm-Widget.lm-Menu.jp-ThemedContainer')
    delete = context_menu.locator('li[data-command="filebrowser:delete"]')
    await delete.click()

    # Click Delete on dialog
    delete_button = dialog.locator('.jp-Dialog-button.jp-mod-accept.jp-mod-warn')
    await delete_button.click()

    # Wait a bit for deleting file
    await expect(file_item).to_be_hidden(timeout=transition_timeout)
    
    print("✓ The notebook for testing is deleted")

await run_pw(_step_delete_notebook)

## Cleanup

In [ ]:
await finish_pw_context()
!rm -rf {work_dir}
print(f"✓ Cleaned up work directory: {work_dir}")

## ✅ All Tests Passed

All lc_multi_outputs extension tests completed successfully:
- Test multi_outputs basic functionality (pin button and output saving)